# 05 실습 2 · 병렬 검색 + 병렬 승인

실습 1은 `retrieve → draft → review → send`가 **한 줄**로 흘렀다. 실습 2에서는 두 곳을 **동시에** 실행한다.

1. **병렬 검색**: 벡터 · 키워드 · 정책 검색 3개를 동시에 돌리고, 결과를 **리듀서**로 합친다.
2. **병렬 승인**: 환불 문의는 **상품 담당**과 **숙소 담당**이 동시에 확인한다. 멈춤(`interrupt`)이 두 개 생긴다.

**그래프 흐름**

    START ─┬→ search_vector  ─┐
           ├→ search_keyword ─┼→ draft ─┬→ check_tour  (멈춤) ─┐
           └→ search_policy  ─┘         └→ check_hotel (멈춤) ─┴→ decide → send / hold → END

| 새로 나오는 것 | 이 실습에서 하는 일 |
|---|---|
| super-step | 같은 단계의 노드는 동시에 실행되고, 서로의 결과를 못 본다 |
| 리듀서 | 같은 키(`docs`, `reviews`)에 동시에 쓰는 값을 합친다 |
| 합류 엣지 `add_edge([...], 노드)` | 갈래가 모두 끝나야 다음 노드를 한 번 실행한다 |
| 병렬 interrupt | 멈춤이 두 개면 **interrupt id별로** 답해야 재개된다 |

- 자료: `travel_docs.csv` (실습 1과 같음)
- 복습: `감기환자 진료 데이터.csv` (맨 아래 복습 미션)

## 1. 준비

In [ ]:
!pip install -q langgraph langchain langchain-openai langchain-community faiss-cpu pandas rank_bm25

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-..."   # 본인 키

## 2. 검색기 3개 만들기

| 검색기 | 방식 | 잘 찾는 것 |
|---|---|---|
| 벡터 (FAISS) | 의미가 비슷한 문서 | "환불돼?" ↔ "환불된다" 같은 표현 차이 |
| 키워드 (BM25) | 글자가 겹치는 문서 | "파도소리", "T15" 같은 고유명사 |
| 정책 (FAISS + 필터) | `category == "정책"`인 문서만 | 환불·취소 규정 |

BM25는 단어를 띄어쓰기로 나누는데, 한국어는 "환불돼?"와 "환불된다"가 다른 단어가 된다. 그래서 **두 글자씩** 잘라서 비교한다.

In [ ]:
import pandas as pd
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_openai import OpenAIEmbeddings

df = pd.read_csv("travel_docs.csv")

docs = []
for _, row in df.iterrows():
    docs.append(Document(
        page_content=f"{row['title']}\n{row['body']}",
        metadata={"doc_id": row["doc_id"], "category": row["category"]},
    ))

db = FAISS.from_documents(docs, OpenAIEmbeddings())


def 두글자(text):
    # "환불돼요" → ["환불", "불돼", "돼요"]
    t = text.replace(" ", "").replace("\n", "")
    return [t[i:i + 2] for i in range(len(t) - 1)]

bm25 = BM25Retriever.from_documents(docs, k=3, preprocess_func=두글자)
print(len(docs), "건 준비 완료")

세 검색기가 같은 질문에 무엇을 가져오는지 비교한다.

In [ ]:
q = "솔바람섬 별빛 2박3일 상품을 출발 3일 전에 취소하면 얼마 환불돼요?"

print("벡터  :", [d.metadata["doc_id"] for d in db.similarity_search(q, k=3)])
print("키워드:", [d.metadata["doc_id"] for d in bm25.invoke(q)])
print("정책  :", [d.metadata["doc_id"] for d in db.similarity_search(q, k=2, filter={"category": "정책"})])

세 검색기의 결과가 조금씩 다르고, **겹치는 문서**도 있다. 합칠 때 중복을 정리해야 한다.

## 3. 실험: 리듀서 없이 병렬로 쓰면?

세 검색 노드가 **같은 단계에서 동시에** `docs`에 값을 쓴다. 합치는 규칙(리듀서)이 없으면 LangGraph가 멈춘다.

In [ ]:
from typing import TypedDict, Annotated, Literal
from operator import add
from langgraph.graph import StateGraph, START, END


class BadState(TypedDict):
    question: str
    docs: list                    # 리듀서 없음


def bad_vector(state):  return {"docs": ["벡터 결과"]}
def bad_keyword(state): return {"docs": ["키워드 결과"]}

bad = StateGraph(BadState)
bad.add_node("vector", bad_vector)
bad.add_node("keyword", bad_keyword)
bad.add_edge(START, "vector")
bad.add_edge(START, "keyword")

try:
    bad.compile().invoke({"question": "환불"})
except Exception as e:
    print(type(e).__name__, ":", str(e)[:90])

## 4. State와 리듀서

| 키 | 쓰는 노드 | 동시에? | 리듀서 |
|---|---|---|---|
| `docs` | 검색 3개 | **예** | `merge_docs` (같은 문서는 하나로, 찾은 검색기 이름은 모음) |
| `trace` | 검색 3개 | **예** | `add` (실행 기록 쌓기) |
| `reviews` | 확인 2개 | **예** | `add` (두 담당자 의견 쌓기) |
| `draft`, `result` | 하나씩 | 아니오 | 없음 (덮어쓰기) |

In [ ]:
def merge_docs(old, new):
    # 같은 doc_id는 하나만 남기고, 어느 검색기가 찾았는지(by)는 모은다
    합친것 = {d["doc_id"]: dict(d) for d in (old or [])}
    for d in new or []:
        if d["doc_id"] in 합친것:
            합친것[d["doc_id"]]["by"] += "+" + d["by"]
        else:
            합친것[d["doc_id"]] = dict(d)
    return list(합친것.values())


class State(TypedDict):
    question: str
    docs: Annotated[list[dict], merge_docs]   # 검색 3개가 동시에 씀
    trace: Annotated[list[str], add]          # 실행 기록
    draft: str
    reviews: Annotated[list[dict], add]       # 확인 2개가 동시에 씀
    result: str

리듀서는 그냥 함수라서 그래프 없이도 테스트할 수 있다.

In [ ]:
a = [{"doc_id": "T15", "text": "...", "by": "vector"}]
b = [{"doc_id": "T15", "text": "...", "by": "keyword"}, {"doc_id": "T16", "text": "...", "by": "keyword"}]
for d in merge_docs(a, b):
    print(d["doc_id"], d["by"])

## 5. 노드 만들기

**검색 노드 3개**: 각자 찾은 것만 돌려준다. 합치기는 리듀서가 한다.
`trace`에 "이 노드가 실행될 때 State에 문서가 몇 개 있었는지"를 남겨 super-step을 눈으로 확인한다.

In [ ]:
def to_dicts(results, by):
    return [{"doc_id": d.metadata["doc_id"], "text": d.page_content, "by": by} for d in results]


# 검색 1: 의미가 비슷한 문서
def search_vector(state):
    found = db.similarity_search(state["question"], k=3)
    return {"docs": to_dicts(found, "vector"),
            "trace": [f"vector(본 문서 {len(state.get('docs', []))}개)"]}


# 검색 2: 글자가 겹치는 문서
def search_keyword(state):
    found = bm25.invoke(state["question"])
    return {"docs": to_dicts(found, "keyword"),
            "trace": [f"keyword(본 문서 {len(state.get('docs', []))}개)"]}


# 검색 3: 정책 문서만
def search_policy(state):
    found = db.similarity_search(state["question"], k=2, filter={"category": "정책"})
    return {"docs": to_dicts(found, "policy"),
            "trace": [f"policy(본 문서 {len(state.get('docs', []))}개)"]}

**초안 노드**: 세 검색이 모두 끝난 뒤 한 번 실행된다. 합쳐진 문서를 본다.

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

PROMPT = """너는 바람길여행 상담원이다.
아래 문서에 있는 내용으로만 고객 질문에 답하고, 없으면 '확인 후 다시 안내드리겠습니다'라고 답해라.
상품 비용과 숙소 비용의 환불 규정이 다르면 나눠서 설명해라.
답 끝에 근거 문서 번호를 [T01]처럼 적어라.

[문서]
{docs}"""


# 초안: 합쳐진 문서만 보고 답변 초안 작성
def draft(state):
    print(f"검색 3개가 찾은 문서 {len(state['docs'])}개로 초안을 쓰고 있습니다.")
    문서 = "\n\n".join(f"[{d['doc_id']}] {d['text']}" for d in state["docs"])
    messages = [("system", PROMPT.format(docs=문서)), ("user", state["question"])]
    return {"draft": llm.invoke(messages).content,
            "trace": [f"draft(본 문서 {len(state['docs'])}개)"]}

**확인 노드 2개**: 같은 단계에서 동시에 실행되고, 각자 `interrupt`로 멈춘다.
`interrupt` 값에 **누가 답해야 하는지(`who`)**를 넣어야 재개할 때 구분할 수 있다.

In [ ]:
from langgraph.types import interrupt, Command


# 확인 1: 상품 담당 (바람길여행 상품 규정 기준)
def check_tour(state):
    답 = interrupt({"who": "상품", "question": state["question"], "draft": state["draft"]})
    return {"reviews": [{"who": "상품", **답}]}


# 확인 2: 숙소 담당 (숙소 운영사 규정 기준)
def check_hotel(state):
    답 = interrupt({"who": "숙소", "question": state["question"], "draft": state["draft"]})
    return {"reviews": [{"who": "숙소", **답}]}

**판정 · 발송 · 보류**: 두 의견이 모두 모이면 판정한다. 둘 다 승인이면 발송, 하나라도 보류면 보류.

In [ ]:
SENT = []   # 발송 기록


# 판정: 두 담당자 의견을 보고 다음 노드 결정
def decide(state) -> Command[Literal["send", "hold"]]:
    모두승인 = all(r["ok"] for r in state["reviews"])
    return Command(goto="send" if 모두승인 else "hold")


# 발송: 둘 다 승인한 경우에만 도달
def send(state):
    SENT.append(state["draft"])
    print("두 담당자 승인 → 고객에게 발송했습니다.")
    return {"result": "발송"}


# 보류: 담당자 메모를 모아 남긴다
def hold(state):
    메모 = " / ".join(f"{r['who']}: {r['memo']}" for r in state["reviews"] if not r["ok"])
    print("보류했습니다 →", 메모)
    return {"result": f"보류 ({메모})"}

## 6. 그래프 조립

- 팬아웃: `START`에서 검색 3개로, `draft`에서 확인 2개로 갈래를 나눈다.
- 합류: `add_edge([...], 노드)`로 **모든 갈래가 끝난 뒤 한 번만** 다음 노드를 실행한다.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

g = StateGraph(State)
for name, fn in [("search_vector", search_vector), ("search_keyword", search_keyword),
                 ("search_policy", search_policy), ("draft", draft),
                 ("check_tour", check_tour), ("check_hotel", check_hotel),
                 ("decide", decide), ("send", send), ("hold", hold)]:
    g.add_node(name, fn)

# 병렬 검색 → 합류
for s in ["search_vector", "search_keyword", "search_policy"]:
    g.add_edge(START, s)
g.add_edge(["search_vector", "search_keyword", "search_policy"], "draft")

# 병렬 확인 → 합류
g.add_edge("draft", "check_tour")
g.add_edge("draft", "check_hotel")
g.add_edge(["check_tour", "check_hotel"], "decide")

g.add_edge("send", END)
g.add_edge("hold", END)

memory = InMemorySaver()
app = g.compile(checkpointer=memory)
app

## 7. 상담 건 ① 상품 + 숙소 환불 문의

"별빛 2박3일" 상품에는 **파도소리 리조트 숙박**이 포함된다. 상품 비용은 바람길여행 규정(T15), 숙소 비용은 숙소 운영사 규정(T16)을 따른다. 두 담당자가 각자 기준으로 확인해야 한다.

In [ ]:
cfg1 = {"configurable": {"thread_id": "상담-101"}}

r = app.invoke({"question": "솔바람섬 별빛 2박3일 상품을 출발 3일 전에 취소하면 얼마 환불돼요?"}, cfg1)

print("\n[실행 기록]", r["trace"])
print("[합쳐진 문서]", [(d["doc_id"], d["by"]) for d in r["docs"]])
print("\n[초안]\n" + r["draft"])

**실행 기록 읽기**

- 검색 3개는 모두 "본 문서 0개"다. 같은 단계라 서로의 결과를 못 본다.
- `draft`는 합쳐진 문서를 본다. 합류 엣지 덕분에 **한 번만** 실행됐다.
- 여러 검색기가 같이 찾은 문서는 `by`에 `vector+keyword`처럼 남는다. 리듀서가 중복을 정리했다.

In [ ]:
ints = r["__interrupt__"]
print("멈춤 개수:", len(ints))
for i in ints:
    print(i.value["who"], "담당 확인 대기 | id:", i.id[:8], "...")

## 8. 실험: 답 하나로 재개하면?

멈춤이 두 개인데 답을 하나만 주면, 어느 담당자의 답인지 알 수 없어 에러가 난다.

In [ ]:
try:
    app.invoke(Command(resume={"ok": True, "memo": ""}), cfg1)
except RuntimeError as e:
    print("RuntimeError:", str(e)[:90])

## 9. interrupt id별로 답하기

`{interrupt id: 답}` 딕셔너리로 재개한다. 순서는 실행마다 바뀔 수 있으니 **`value["who"]`로 구분**한다.

숙소 담당이 초안을 보니, 숙소 비용 환불(체크인 3~6일 전 50%) 안내가 부정확하다고 보류했다고 하자.

In [ ]:
답들 = {}
for i in ints:
    if i.value["who"] == "상품":
        답들[i.id] = {"ok": True, "memo": "상품 30% 환불 안내 맞음"}
    else:
        답들[i.id] = {"ok": False, "memo": "숙소 비용은 체크인 3일 전이면 50% 환불, 따로 안내 필요"}

r = app.invoke(Command(resume=답들), cfg1)
print("\n[결과]", r["result"])
print("[의견]", [(x["who"], x["ok"]) for x in r["reviews"]])
print("발송 건수:", len(SENT))

## 10. 상담 건 ② 둘 다 승인 → 발송

In [ ]:
cfg2 = {"configurable": {"thread_id": "상담-102"}}

r = app.invoke({"question": "파도소리 리조트 체크인 7일 전에 취소하면 환불돼요?"}, cfg2)
print("\n[초안]\n" + r["draft"])

답들 = {i.id: {"ok": True, "memo": "확인"} for i in r["__interrupt__"]}
r = app.invoke(Command(resume=답들), cfg2)
print("\n[결과]", r["result"], "| 발송 건수:", len(SENT))

## 11. 기록 보기

`get_state_history`로 보면 **같은 단계에서 실행된 노드들이 한 줄에** 묶여 있다. 이것이 super-step이다.

In [ ]:
for s in app.get_state_history(cfg1):
    print(s.next)

In [ ]:
print("상담 건:", {c.config["configurable"]["thread_id"] for c in memory.list(None)})

## 12. 정리

| 개념 | 핵심 |
|---|---|
| super-step | 같은 단계의 노드는 동시에 실행되고, 같은 State(이전 단계 결과)를 본다 |
| 리듀서 | 같은 키에 동시에 쓰면 필수. 없으면 `InvalidUpdateError` |
| 합류 엣지 | `add_edge([a, b, c], 다음)` → 모두 끝난 뒤 한 번 |
| 병렬 interrupt | 멈춤이 여러 개면 `Command(resume={id: 답, ...})` |
| `who` 넣기 | interrupt 순서는 정해져 있지 않다. 값 안에 누구 몫인지 넣어 구분한다 |

> 실습 1: **한 사람이 순서대로** 확인한다. 실습 2: **여러 곳을 동시에** 검색하고, **여러 사람이 동시에** 확인한다.

---
# 복습 미션 · 감기환자 진료 데이터

`감기환자 진료 데이터.csv`로 같은 구조를 만든다.

**상황**: 비슷한 과거 진료를 **두 가지 방법으로 동시에** 검색하고, 소견 초안을 **담당의(진단)와 약사(처방)가 동시에** 확인한다. 둘 다 승인해야 진료 기록에 저장한다.

    START ─┬→ search_vector  ─┐
           └→ search_keyword ─┴→ draft ─┬→ check_doctor     (멈춤) ─┐
                                        └→ check_pharmacist (멈춤) ─┴→ decide → save / hold → END

| 미션 | 할 일 |
|---|---|
| M1 | 진료 기록 Document + FAISS + BM25(두글자) 만들고, `"6세 여아 재채기 가래"`로 두 검색 결과 비교 |
| M2 | `merge_docs`를 `환자ID` 기준으로 바꾸고 단독 테스트 |
| M3 | 노드 7개 + 그래프 조립 (`send` → `save`, 확인 노드는 의사·약사) |
| M4 | 환자-A: 답 하나로 재개 → 에러 확인 → id별로 답하기 (의사 승인, 약사 보류) |
| M5 | 환자-B: 둘 다 승인 → 저장, `trace`와 `get_state_history`로 super-step 확인 |

### M1 · 검색기 2개

In [ ]:
df2 = pd.read_csv("감기환자 진료 데이터.csv")

# TODO: records (page_content=진료내용, metadata=환자ID·진단명·처방)
# TODO: db2 = FAISS..., bm25_2 = BM25Retriever...(preprocess_func=두글자)
# TODO: "6세 여아 재채기 가래"로 두 검색기의 환자ID 비교

### M2 · 리듀서

In [ ]:
# TODO: merge_records(old, new) - 환자ID 기준 중복 제거, by 모으기
# TODO: 단독 테스트

### M3 · 그래프

In [ ]:
# TODO: State2 (question, docs, trace, draft, reviews, result)
# TODO: search_vector2, search_keyword2, draft2, check_doctor, check_pharmacist, decide2, save, hold2
# TODO: 조립 + memory2로 compile → app2

### M4 · 환자-A: id별로 답하기

In [ ]:
# TODO: "38세 남성, 7일째 오한과 기침, 체온 39.1도" 실행
# TODO: 답 하나로 재개 → RuntimeError 확인
# TODO: 의사 승인, 약사 보류("해열제와 진해거담제 함께 검토 필요")로 재개 → 결과 확인

### M5 · 환자-B: 둘 다 승인 + super-step 확인

In [ ]:
# TODO: "6세 여아, 재채기와 가래, 37.9도" → 둘 다 승인 → 저장
# TODO: trace 출력, get_state_history(cfgB)의 next 출력

---
### 예시 답안 (막힐 때만 펼쳐 보기)

In [ ]:
# M1
records = []
for _, row in df2.iterrows():
    records.append(Document(
        page_content=row["진료내용"],
        metadata={"환자ID": row["환자ID"], "진단명": row["진단명"], "처방": row["처방"]},
    ))

db2 = FAISS.from_documents(records, OpenAIEmbeddings())
bm25_2 = BM25Retriever.from_documents(records, k=3, preprocess_func=두글자)

q = "6세 여아 재채기 가래"
print("벡터  :", [d.metadata["환자ID"] for d in db2.similarity_search(q, k=3)])
print("키워드:", [d.metadata["환자ID"] for d in bm25_2.invoke(q)])

In [ ]:
# M2
def merge_records(old, new):
    합친것 = {d["환자ID"]: dict(d) for d in (old or [])}
    for d in new or []:
        if d["환자ID"] in 합친것:
            합친것[d["환자ID"]]["by"] += "+" + d["by"]
        else:
            합친것[d["환자ID"]] = dict(d)
    return list(합친것.values())

print(merge_records([{"환자ID": "P027", "text": "...", "by": "vector"}],
                    [{"환자ID": "P027", "text": "...", "by": "keyword"},
                     {"환자ID": "P011", "text": "...", "by": "keyword"}]))

In [ ]:
# M3
PROMPT2 = """너는 진료 보조 AI다.
아래 과거 진료 기록만 참고해 예상 진단과 처방을 제안하고, 근거 환자ID를 [P001]처럼 적어라.
최종 판단은 의사가 한다는 문장을 마지막에 붙여라.

[과거 진료 기록]
{docs}"""

SAVED = []


class State2(TypedDict):
    question: str
    docs: Annotated[list[dict], merge_records]
    trace: Annotated[list[str], add]
    draft: str
    reviews: Annotated[list[dict], add]
    result: str


def rec_dicts(results, by):
    return [{"환자ID": d.metadata["환자ID"], "text": d.page_content, "by": by} for d in results]


# 검색 1: 의미가 비슷한 진료 기록
def search_vector2(state):
    return {"docs": rec_dicts(db2.similarity_search(state["question"], k=3), "vector"),
            "trace": [f"vector(본 기록 {len(state.get('docs', []))}개)"]}


# 검색 2: 글자가 겹치는 진료 기록
def search_keyword2(state):
    return {"docs": rec_dicts(bm25_2.invoke(state["question"]), "keyword"),
            "trace": [f"keyword(본 기록 {len(state.get('docs', []))}개)"]}


# 초안: 합쳐진 기록으로 소견 작성
def draft2(state):
    기록 = "\n".join(f"[{d['환자ID']}] {d['text']}" for d in state["docs"])
    messages = [("system", PROMPT2.format(docs=기록)), ("user", state["question"])]
    return {"draft": llm.invoke(messages).content,
            "trace": [f"draft(본 기록 {len(state['docs'])}개)"]}


# 확인 1: 담당의 (진단)
def check_doctor(state):
    답 = interrupt({"who": "의사", "question": state["question"], "draft": state["draft"]})
    return {"reviews": [{"who": "의사", **답}]}


# 확인 2: 약사 (처방)
def check_pharmacist(state):
    답 = interrupt({"who": "약사", "question": state["question"], "draft": state["draft"]})
    return {"reviews": [{"who": "약사", **답}]}


# 판정: 둘 다 승인이면 저장, 아니면 보류
def decide2(state) -> Command[Literal["save", "hold"]]:
    return Command(goto="save" if all(r["ok"] for r in state["reviews"]) else "hold")


# 저장: 둘 다 승인한 소견만
def save(state):
    SAVED.append(state["draft"])
    print("의사·약사 승인 → 진료 기록에 저장했습니다.")
    return {"result": "저장"}


# 보류: 반려 메모를 남긴다
def hold2(state):
    메모 = " / ".join(f"{r['who']}: {r['memo']}" for r in state["reviews"] if not r["ok"])
    print("보류했습니다 →", 메모)
    return {"result": f"보류 ({메모})"}


g2 = StateGraph(State2)
for name, fn in [("search_vector", search_vector2), ("search_keyword", search_keyword2),
                 ("draft", draft2), ("check_doctor", check_doctor),
                 ("check_pharmacist", check_pharmacist), ("decide", decide2),
                 ("save", save), ("hold", hold2)]:
    g2.add_node(name, fn)
g2.add_edge(START, "search_vector")
g2.add_edge(START, "search_keyword")
g2.add_edge(["search_vector", "search_keyword"], "draft")
g2.add_edge("draft", "check_doctor")
g2.add_edge("draft", "check_pharmacist")
g2.add_edge(["check_doctor", "check_pharmacist"], "decide")
g2.add_edge("save", END)
g2.add_edge("hold", END)

memory2 = InMemorySaver()
app2 = g2.compile(checkpointer=memory2)
app2

In [ ]:
# M4
cfgA = {"configurable": {"thread_id": "환자-A"}}
r = app2.invoke({"question": "38세 남성, 7일째 오한과 기침, 체온 39.1도"}, cfgA)
print("\n[초안]\n" + r["draft"])

try:
    app2.invoke(Command(resume={"ok": True, "memo": ""}), cfgA)
except RuntimeError as e:
    print("\nRuntimeError:", str(e)[:90])

답들 = {}
for i in r["__interrupt__"]:
    if i.value["who"] == "의사":
        답들[i.id] = {"ok": True, "memo": "진단 동의"}
    else:
        답들[i.id] = {"ok": False, "memo": "해열제와 진해거담제 함께 검토 필요"}

r = app2.invoke(Command(resume=답들), cfgA)
print("\n[결과]", r["result"], "| 저장 건수:", len(SAVED))

In [ ]:
# M5
cfgB = {"configurable": {"thread_id": "환자-B"}}
r = app2.invoke({"question": "6세 여아, 재채기와 가래, 37.9도"}, cfgB)
r = app2.invoke(Command(resume={i.id: {"ok": True, "memo": "확인"} for i in r["__interrupt__"]}), cfgB)
print("\n[결과]", r["result"], "| 저장 건수:", len(SAVED))
print("[실행 기록]", r["trace"])

for s in app2.get_state_history(cfgB):
    print(s.next)